<a href="https://colab.research.google.com/github/lqh2005/TinyProject/blob/main/download_raw_cases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount Google Drive & Import Libraries

In [ ]:
# ==========================================\n",
# CELL 1: Connect Google Drive & Install Libraries
# ==========================================\n",
import os
import json
import time
import requests
from bs4 import BeautifulSoup
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

print("Successfully connected to Google Drive!")

Mounted at /content/drive
Successfully connected to Google Drive!


Variable Configuration & Folder Structure Setup

In [ ]:
# ==========================================
# CELL 2: Configuration & Folder Setup
# ==========================================

# 1. Member information configuration
MEMBER_NAME = "Ho Trung Quan"

# Base directory path for storing raw data on Google Drive
BASE_DIR = f"/content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_{MEMBER_NAME}_raw"

# 2. Disease list and target configurations
DISEASE_CONFIGS = [
    {
        "name": "Ocular Toxoplasmosis",
        "code": "OCULAR_TOXO",
        "is_rare_case": False,
        "target": 50,
        "query": '("Ocular Toxoplasmosis" OR "Toxoplasmic retinochoroiditis") AND "case report" HAS_FT:y OPEN_ACCESS:y'
    },
    {
        "name": "Infectious Skin Lesions",
        "code": "SKIN_LESION",
        "is_rare_case": False,
        "target": 50,
        "query": '("Leishmaniasis" OR "Tinea" OR "Cutaneous infection") AND "case report" HAS_FT:y OPEN_ACCESS:y'
    }
]

# 3. Automatic directory structure creation
os.makedirs(BASE_DIR, exist_ok=True)

for config in DISEASE_CONFIGS:
    disease_dir = os.path.join(BASE_DIR, config["name"])
    img_dir = os.path.join(disease_dir, "raw_images")
    os.makedirs(img_dir, exist_ok=True)
    print(f"✓ Directory ready: {img_dir}")

✓ Directory ready: /content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_Ho Trung Quan_raw/Ocular Toxoplasmosis/raw_images
✓ Directory ready: /content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_Ho Trung Quan_raw/Infectious Skin Lesions/raw_images


Helper Function for Automatic Data Collection via Europe PMC API

In [ ]:
# ==========================================\n",
# CELL 3: Core Scraper Functions (Europe PMC API)
# ==========================================\n",

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

def search_europe_pmc(query, limit=100):
    """
    Search Open Access case report articles from Europe PMC API
    """
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    params = {
        'query': query,
        'format': 'json',
        'pageSize': limit,
        'resultType': 'core'
    }
    try:
        response = requests.get(url, params=params, headers=HEADERS, timeout=15)
        if response.status_code == 200:
            data = response.json()
            return data.get('resultList', {}).get('result', [])
    except Exception as e:
        print(f"Error searching Europe PMC: {e}")
    return []

def extract_case_details(pmcid):
    """
    Extract case text (Case Report/Presentation) and raw image URLs from PMC article
    """
    pmc_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/"
    try:
        res = requests.get(pmc_url, headers=HEADERS, timeout=15)
        if res.status_code != 200:
            return None, None, None

        soup = BeautifulSoup(res.content, 'html.parser')

        # 1. Extract text (Find Case Presentation / Abstract / Body section)
        case_text = ""
        # Try finding Case Presentation or Case Report section in the article
        case_sec = soup.find(lambda tag: tag.name in ['div', 'section'] and 'case' in tag.get('id', '').lower())
        if not case_sec:
            case_sec = soup.find(lambda tag: tag.name in ['h2', 'h3'] and 'case' in tag.text.lower())
            if case_sec:
                case_sec = case_sec.find_parent(['div', 'section'])

        if case_sec:
            paragraphs = case_sec.find_all('p')
            case_text = " ".join([p.text.strip() for p in paragraphs if len(p.text.strip()) > 30])

        # Fallback: If no specific section is found, extract Abstract or main body paragraphs
        if len(case_text) < 100:
            abstract_div = soup.find('div', {'class': 'abstract'}) or soup.find('abstract')
            if abstract_div:
                case_text = abstract_div.text.strip()
            else:
                paragraphs = soup.find_all('p')
                case_text = " ".join([p.text.strip() for p in paragraphs[:5] if len(p.text.strip()) > 30])

        if not case_text or len(case_text) < 50:
            return None, None, None

        # 2. Extract representative raw medical image
        image_url = None
        # Extract images from figure / img tags in PMC article
        figures = soup.find_all('figure')
        for fig in figures:
            img_tag = fig.find('img')
            if img_tag and img_tag.get('src'):
                src = img_tag['src']
                if not src.startswith('http'):
                    src = f"https://www.ncbi.nlm.nih.gov{src}" if src.startswith('/') else f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/{src}"
                # Filter out small icons or logos
                if 'bin' in src or 'jpg' in src or 'png' in src or 'figure' in src.lower():
                    image_url = src
                    break

        if not image_url:
            # Find any large image tag
            for img in soup.find_all('img'):
                src = img.get('src', '')
                if 'mc' in src.lower() or 'fig' in src.lower() or 'gr' in src.lower():
                    if not src.startswith('http'):
                        src = f"https://www.ncbi.nlm.nih.gov{src}"
                    image_url = src
                    break

        return pmc_url, case_text, image_url

    except Exception as e:
        return None, None, None

Automated Collection & Storage Loop (With Deduplication Mechanism)

In [ ]:
# ==========================================\n",
# CELL 4: Automated Data Collection Pipeline (Robust Deduplication)
# ==========================================\n",

def download_image(url, save_path):
    """Download and save raw image from URL"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code == 200 and len(resp.content) > 5000: # Ensure valid image file (>5KB)
            with open(save_path, 'wb') as f:
                f.write(resp.content)
            return True
    except Exception as e:
        pass
    return False

print(f"=== STARTING RAW DATA COLLECTION FOR PROJECT MultiCARE ({MEMBER_NAME}) ===")

for disease in DISEASE_CONFIGS:
    disease_name = disease["name"]
    disease_code = disease["code"]
    target_count = disease["target"]
    is_rare = disease["is_rare_case"]

    disease_dir = os.path.join(BASE_DIR, disease_name)
    img_dir = os.path.join(disease_dir, "raw_images")
    json_path = os.path.join(disease_dir, "raw_cases.json")

    # 1. READ PREVIOUSLY DOWNLOADED CASES FROM EXISTING JSON FILE (IF ANY)
    cases_data = []
    collected_urls = set()

    if os.path.exists(json_path):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                cases_data = json.load(f)
                for item in cases_data:
                    if "source_url" in item and item["source_url"]:
                        # Normalize URL for comparison (remove trailing slash if any)
                        collected_urls.add(item["source_url"].strip().rstrip('/'))
        except Exception as e:
            print(f"⚠ Could not read existing JSON file ({e}), initializing fresh state.")
            cases_data = []

    current_count = len(cases_data)
    print(f"\n---> Disease: {disease_name} (Current: {current_count}/{target_count})")

    if current_count >= target_count:
        print(f"✓ Already reached target of {target_count} cases for disease {disease_name}. Skipping!")
        continue

    # 2. SEARCH ARTICLES FROM EUROPE PMC API
    articles = search_europe_pmc(disease["query"], limit=300)
    print(f"Found {len(articles)} potential articles from Europe PMC...")

    skipped_count = 0

    for article in articles:
        if current_count >= target_count:
            break

        pmcid = article.get('pmcid')
        if not pmcid:
            continue

        # Construct standard base URL
        pmc_url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/".rstrip('/')

        # 🛑 DEDUPLICATION 1: Check if URL already exists in JSON file
        if pmc_url in collected_urls:
            skipped_count += 1
            continue

        # Extract article details
        source_url, raw_text, image_url = extract_case_details(pmcid)

        # Skip if article lacks text or image
        if not raw_text or not image_url:
            continue

        # 🛑 DEDUPLICATION 2: Re-check URL returned from extraction function
        clean_source_url = source_url.strip().rstrip('/') if source_url else pmc_url
        if clean_source_url in collected_urls:
            skipped_count += 1
            continue

        # 3. NAME CASE AND IMAGE FILE ACCORDING TO NEXT INDEX
        case_index = current_count + 1
        case_id = f"MC_{disease_code}_{case_index:03d}"
        img_filename = f"{case_id}.jpg"
        img_save_path = os.path.join(img_dir, img_filename)

        # 4. DOWNLOAD IMAGE AND UPDATE DATA
        if download_image(image_url, img_save_path):
            case_record = {
                "case_id": case_id,
                "disease_category": disease_name,
                "is_rare_case": is_rare,
                "source_url": source_url,
                "raw_image_filename": img_filename,
                "raw_case_text": raw_text,
                "annotator_id": MEMBER_NAME
            }

            # Append to list and save to disk immediately to prevent data loss on network drops
            cases_data.append(case_record)
            collected_urls.add(clean_source_url)
            current_count += 1

            with open(json_path, 'w', encoding='utf-8') as f:
                json.dump(cases_data, f, ensure_ascii=False, indent=2)

            print(f"[{disease_name} {current_count}/{target_count}] Successfully downloaded {img_filename}")
            time.sleep(0.5)  # Prevent hitting rate limits
        else:
            continue

    if skipped_count > 0:
        print(f"ℹ️ Skipped {skipped_count} articles that were already collected or did not meet requirements.")

    print(f"✓ Results for this run ({disease_name}): {current_count}/{target_count} cases.")

print("\n==================================================")
print("🎉 RAW DATA COLLECTION COMPLETED!")
print(f"Directory structure saved at: {BASE_DIR}")

=== STARTING RAW DATA COLLECTION FOR PROJECT MultiCARE (Ho Trung Quan) ===

---> Disease: Ocular Toxoplasmosis (Current: 50/50)
✓ Already reached target of 50 cases for disease Ocular Toxoplasmosis. Skipping!

---> Disease: Infectious Skin Lesions (Current: 40/50)
Found 300 potential articles from Europe PMC...
[Infectious Skin Lesions 41/50] Successfully downloaded MC_SKIN_LESION_041.jpg
[Infectious Skin Lesions 42/50] Successfully downloaded MC_SKIN_LESION_042.jpg
[Infectious Skin Lesions 43/50] Successfully downloaded MC_SKIN_LESION_043.jpg
[Infectious Skin Lesions 44/50] Successfully downloaded MC_SKIN_LESION_044.jpg
[Infectious Skin Lesions 45/50] Successfully downloaded MC_SKIN_LESION_045.jpg
[Infectious Skin Lesions 46/50] Successfully downloaded MC_SKIN_LESION_046.jpg
[Infectious Skin Lesions 47/50] Successfully downloaded MC_SKIN_LESION_047.jpg
[Infectious Skin Lesions 48/50] Successfully downloaded MC_SKIN_LESION_048.jpg
[Infectious Skin Lesions 49/50] Successfully downloaded

Verification & Summary Check

In [ ]:
# ==========================================
# CELL 5: Verification & Summary Check
# ==========================================
import glob

print("=== RAW DATA COLLECTION SUMMARY REPORT ===")

for disease in DISEASE_CONFIGS:
    d_name = disease["name"]
    d_dir = os.path.join(BASE_DIR, d_name)
    img_dir = os.path.join(d_dir, "raw_images")
    json_file = os.path.join(d_dir, "raw_cases.json")

    images_count = len(glob.glob(os.path.join(img_dir, "*.jpg")))

    json_count = 0
    if os.path.exists(json_file):
        with open(json_file, 'r', encoding='utf-8') as f:
            json_count = len(json.load(f))

    print(f"\n📂 Disease: {d_name}")
    print(f"   - Number of images (.jpg): {images_count} / {disease['target']}")
    print(f"   - Number of cases in raw_cases.json: {json_count} / {disease['target']}")

    if images_count == disease['target'] and json_count == disease['target']:
        print(f"   STATUS: OK (100% target reached)")
    else:
        print(f"   STATUS: INCOMPLETE TARGET (Need to re-run Cell 4)")

=== RAW DATA COLLECTION SUMMARY REPORT ===

📂 Disease: Ocular Toxoplasmosis
   - Number of images (.jpg): 50 / 50
   - Number of cases in raw_cases.json: 50 / 50
   STATUS: OK (100% target reached)

📂 Disease: Infectious Skin Lesions
   - Number of images (.jpg): 50 / 50
   - Number of cases in raw_cases.json: 50 / 50
   STATUS: OK (100% target reached)


Compress Disease Folders into ZIP

In [ ]:
# ==========================================
# CELL 6: Compress Directory into ZIP
# ==========================================
import os
import shutil

parent_dir = f"/content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_{MEMBER_NAME}_raw"
output_zip_path = f"/content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_{MEMBER_NAME}_raw"

if os.path.exists(parent_dir):
    print(f"Checking subfolders inside: {parent_dir}")
    subfolders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]
    print(f"Found folders: {subfolders}")

    print("Compressing data, please wait a moment...")
    shutil.make_archive(output_zip_path, 'zip', parent_dir)
    print(f"✅ Compression completed! ZIP file created at: {output_zip_path}.zip")
else:
    print(f"❌ Path does not exist: {parent_dir}")

Checking subfolders inside: /content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_Ho Trung Quan_raw
Found folders: ['Ocular Toxoplasmosis', 'Infectious Skin Lesions']
Compressing data, please wait a moment...
✅ Compression completed! ZIP file created at: /content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_Ho Trung Quan_raw.zip


PREPROCESSING & LLM-ASSISTED LABELING (MELANOMA & CATARACT)

In [ ]:
# ==============================================================================
# ==============================================================================
# KHÔNG DÙNG CELL NÀY
# ==============================================================================
# ==============================================================================






# ==============================================================================
# CELL 7: PREPROCESSING & LLM-ASSISTED LABELING (FAST SSD READ & AUTO-RESUME)
# ==============================================================================

import os
import re
import json
import torch
import glob
import shutil
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

# ------------------------------------------------------------------------------
# 0. CẤU HÌNH ĐƯỜNG DẪN & GIẢI NÉN DATASET SANG SSD COLAB
# ------------------------------------------------------------------------------
MEMBER_NAME = "Ho Trung Quan"

# Đường dẫn file ZIP và thư mục backup trên Google Drive
ZIP_DRIVE_PATH = f"/content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_{MEMBER_NAME}_raw.zip"
DRIVE_BACKUP_DIR = f"/content/drive/MyDrive/Small multimodal models for clinical diagnosis/Raw dataset/New/data_{MEMBER_NAME}_raw"

# Đường dẫn đĩa SSD cục bộ của Colab để đọc file cực nhanh
LOCAL_EXTRACT_DIR = "/content/dataset"
BASE_DIR = os.path.join(LOCAL_EXTRACT_DIR, f"data_{MEMBER_NAME}_raw") if os.path.exists(os.path.join(LOCAL_EXTRACT_DIR, f"data_{MEMBER_NAME}_raw")) else LOCAL_EXTRACT_DIR

# Giải nén từ Drive sang SSD Colab nếu file ZIP tồn tại
if os.path.exists(ZIP_DRIVE_PATH):
    print("📦 Đang giải nén file ZIP từ Drive sang đĩa SSD tạm của Colab...")
    shutil.unpack_archive(ZIP_DRIVE_PATH, LOCAL_EXTRACT_DIR, 'zip')
    print("✅ Giải nén hoàn tất! Đã sẵn sàng đọc dữ liệu từ SSD.")
elif os.path.exists(DRIVE_BACKUP_DIR):
    print("⚠️ Không tìm thấy file ZIP, sẽ đọc trực tiếp từ thư mục Drive.")
    BASE_DIR = DRIVE_BACKUP_DIR

DISEASE_SPECS = {
    "Melanoma": {
        "disease_code": "MELANOMA",
        "modality": "Dermoscopy / Clinical Photo",
        "is_rare": False
    },
    "Cataract": {
        "disease_code": "CATARACT",
        "modality": "Fundus / Eye Photo",
        "is_rare": False
    }
}

# ==============================================================================
# BƯỚC 1: HÀM PREPROCESSING & VALIDATION
# ==============================================================================

def clean_text(text: str) -> str:
    """Loại bỏ ký tự thừa, HTML tags, khoảng trắng thừa"""
    if not text:
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_and_parse_json(response_text: str) -> dict:
    """Xử lý ngoại lệ bóc tách khối JSON từ phản hồi của LLM"""
    try:
        match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if match:
            json_str = match.group(0)
            return json.loads(json_str)
        return None
    except Exception as e:
        print(f"❌ Lỗi parse JSON: {e}")
        return None

# ==============================================================================
# BƯỚC 2: KHỞI TẠO MÔ HÌNH LLM OPEN-SOURCE (Qwen2.5-7B-Instruct)
# ==============================================================================
print("🚀 Đang tải mô hình LLM Open-Source (Qwen2.5-7B-Instruct)...")

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

SYSTEM_PROMPT = """You are an expert Clinical Data Extraction Assistant specializing in processing open-access medical case reports into standardized JSON structures for multimodal medical AI research.
YOUR TASK: Read the provided raw medical case report text and extract structured information strictly conforming to the JSON schema below. STRICT RULES & CONSTRAINTS:
1. JSON ONLY: Output MUST be a single valid JSON object. Do not include any introductory text, preambles, markdown formatting outside json, or conversational meta-text.
2. NO HALLUCINATION: Extract only factual information present in the input report. If a detail is missing, set its value to null or "Unknown". Do NOT invent medical facts.
3. LANGUAGE: All string values MUST be in clear English.
4. EXACT SCHEMA COMPLIANCE:

JSON SCHEMA:
{
  "case_id": "string",
  "disease_category": "string",
  "is_rare_case": boolean,
  "modality": "string",
  "patient_info": {
    "age": integer_or_null,
    "gender": "Male" | "Female" | "Unknown"
  },
  "clinical_history": "A clear concise summary of patient demographics, presenting symptoms, duration, and clinical background.",
  "findings": "Detailed key imaging and physical diagnostic findings extracted from the report.",
  "ground_truth_diagnosis": "The exact final diagnosis confirmed by doctors.",
  "diagnostic_rationale": "The official reasoning, evidence, lab test, or pathology result.",
  "image_filename": "string",
  "source_url": "string",
  "annotator_id": "string"
}"""

def run_llm_extraction(raw_text: str, meta_info: dict) -> dict:
    """Gửi prompt cho LLM và trích xuất JSON"""
    user_prompt = f"""Please extract the clinical case details into JSON format based on system guidelines. Meta Inputs:
- Case ID:
{meta_info['case_id']}
- Disease Category:
{meta_info['disease_category']}
- Is Rare Case:
{str(meta_info['is_rare_case']).lower()}
- Modality:
{meta_info['modality']}
- Image Filename:
{meta_info['image_filename']}
- Source URL:
{meta_info['source_url']}
- Annotator ID:
{meta_info['annotator_id']}

Raw Case Report Text:
-------------------
{raw_text}
-------------------"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=False
        )

    response_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0]

# ==============================================================================
# BƯỚC 3: XỬ LÝ NỐI TIẾP (AUTO-RESUME) & LƯU ĐỒNG THỜI CẢ SSD VÀ DRIVE
# ==============================================================================

for disease_name, specs in DISEASE_SPECS.items():
    print(f"\n==================================================")
    print(f"🔄 ĐANG XỬ LÝ DỮ LIỆU BỆNH: {disease_name.upper()}")
    print(f"==================================================")

    # Thư mục đọc trên SSD
    disease_dir_local = os.path.join(BASE_DIR, disease_name)
    raw_json_path = os.path.join(disease_dir_local, "raw_cases.json")
    processed_labels_local_path = os.path.join(disease_dir_local, "labels.json")

    # Thư mục backup ghi trên Drive
    disease_dir_drive = os.path.join(DRIVE_BACKUP_DIR, disease_name)
    processed_labels_drive_path = os.path.join(disease_dir_drive, "labels.json")
    os.makedirs(disease_dir_drive, exist_ok=True)

    if not os.path.exists(raw_json_path):
        print(f"⚠️ Không tìm thấy raw_cases.json tại {raw_json_path}. Bỏ qua!")
        continue

    # 1. TẢI CÁC CASE ĐÃ GÁN NHÃN TRƯỚC ĐÓ (NẾU CÓ) ĐỂ CHẠY NỐI TIẾP
    processed_cases = []
    processed_ids = set()

    # Thử đọc từ SSD local trước, nếu chưa có thì đọc từ Drive backup
    target_labels_path = processed_labels_local_path if os.path.exists(processed_labels_local_path) else processed_labels_drive_path

    if os.path.exists(target_labels_path):
        try:
            with open(target_labels_path, 'r', encoding='utf-8') as f:
                processed_cases = json.load(f)
                processed_ids = {item["case_id"] for item in processed_cases if "case_id" in item}
            print(f"ℹ️ Đã tìm thấy labels.json cũ: Đã xử lý {len(processed_ids)} ca trước đó. Sẽ tự động nhảy đến ca tiếp theo!")
        except Exception as e:
            print(f"⚠️ Lỗi đọc labels.json cũ, tạo mới list: {e}")
            processed_cases = []

    # 2. ĐỌC DỮ LIỆU THÔ RAW CASES
    with open(raw_json_path, 'r', encoding='utf-8') as f:
        raw_cases = json.load(f)

    skipped_count = 0

    for idx, case in enumerate(raw_cases, start=1):
        case_id = case.get("case_id", f"MC_{specs['disease_code']}_{idx:03d}")

        # 🛑 KIỂM TRA TỰ ĐỘNG BỎ QUA CASE ĐÃ XỬ LÝ
        if case_id in processed_ids:
            skipped_count += 1
            continue

        print(f"\n[Progress: {len(processed_cases) + 1}/{len(raw_cases)}] Processing Case ID: {case_id}")

        # Preprocessing text
        cleaned_text = clean_text(case.get("raw_case_text", ""))
        if len(cleaned_text) < 50:
            print(f"⚠️️ Văn bản quá ngắn (<50 ký tự). Bỏ qua {case_id}")
            continue

        # Meta Data
        meta_info = {
            "case_id": case_id,
            "disease_category": disease_name,
            "is_rare_case": specs["is_rare"],
            "modality": specs["modality"],
            "image_filename": case.get("raw_image_filename", f"{case_id}.jpg"),
            "source_url": case.get("source_url", ""),
            "annotator_id": MEMBER_NAME
        }

        # LLM Inference
        raw_llm_response = run_llm_extraction(cleaned_text, meta_info)
        parsed_json = clean_and_parse_json(raw_llm_response)

        if parsed_json:
            # Ghi đè thông tin định danh chính xác
            parsed_json["case_id"] = meta_info["case_id"]
            parsed_json["disease_category"] = meta_info["disease_category"]
            parsed_json["is_rare_case"] = meta_info["is_rare_case"]
            parsed_json["modality"] = meta_info["modality"]
            parsed_json["image_filename"] = meta_info["image_filename"]
            parsed_json["source_url"] = meta_info["source_url"]
            parsed_json["annotator_id"] = meta_info["annotator_id"]

            processed_cases.append(parsed_json)
            processed_ids.add(case_id)

            # 1. Lưu file labels.json cục bộ trên SSD Colab
            with open(processed_labels_local_path, 'w', encoding='utf-8') as f:
                json.dump(processed_cases, f, ensure_ascii=False, indent=2)

            # 2. Đồng thời lưu 1 bản trực tiếp về Google Drive để bảo vệ dữ liệu tuyệt đối
            with open(processed_labels_drive_path, 'w', encoding='utf-8') as f:
                json.dump(processed_cases, f, ensure_ascii=False, indent=2)

            print(f"✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: {case_id}")
        else:
            print(f"❌ Không thể parse JSON cho case {case_id}")

    if skipped_count > 0:
        print(f"ℹ️ Đã tự động bỏ qua {skipped_count} ca đã được gán nhãn từ trước.")

    print(f"🎉 Hoàn thành bệnh {disease_name}: Hiện có {len(processed_cases)}/{len(raw_cases)} ca đã được lưu tại {processed_labels_drive_path}")

📦 Đang giải nén file ZIP từ Drive sang đĩa SSD tạm của Colab...
✅ Giải nén hoàn tất! Đã sẵn sàng đọc dữ liệu từ SSD.
🚀 Đang tải mô hình LLM Open-Source (Qwen2.5-7B-Instruct)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🔄 ĐANG XỬ LÝ DỮ LIỆU BỆNH: MELANOMA
ℹ️ Đã tìm thấy labels.json cũ: Đã xử lý 38 ca trước đó. Sẽ tự động nhảy đến ca tiếp theo!

[Progress: 39/50] Processing Case ID: MC_MELANOMA_029
❌ Lỗi parse JSON: Expecting ',' delimiter: line 9 column 4 (char 199)
❌ Không thể parse JSON cho case MC_MELANOMA_029

[Progress: 39/50] Processing Case ID: MC_MELANOMA_040
✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: MC_MELANOMA_040

[Progress: 40/50] Processing Case ID: MC_MELANOMA_041
✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: MC_MELANOMA_041

[Progress: 41/50] Processing Case ID: MC_MELANOMA_042
✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: MC_MELANOMA_042

[Progress: 42/50] Processing Case ID: MC_MELANOMA_043
✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: MC_MELANOMA_043

[Progress: 43/50] Processing Case ID: MC_MELANOMA_044
✅ Đã gán nhãn & lưu đồng bộ (SSD + Drive) thành công: MC_MELANOMA_044

[Progress: 44/50] Processing Case ID: MC_MELANOMA_045
✅ Đã gán nhãn & lưu 